# Actividad práctica: SQL, pandas y el agente de la Pizzería Don Mario

**Alumno:** Josué Gutierrez   
**Código:** 2310128

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from don_mario_lab import crear_mini_base, crear_base, experimento_escala, torneo

# Configuración del código personal
MI_CODIGO = 20261234

## Parte 1 a 4: Calentamiento SQL vs Pandas

In [ ]:
# Crear la mini base para calentamiento
mini = crear_mini_base()

### Preguntas 1 a 5 usando pd.read_sql
*(Nota: ajusta las consultas según las tablas y columnas exactas creadas por crear_mini_base)*

In [ ]:
# Pregunta 1
query_1 = "SELECT * FROM ventas LIMIT 5;"
df_1 = pd.read_sql(query_1, mini)
display(df_1)

# Pregunta 2 a 5 (Ejemplos de espacio de trabajo)
# query_2 = ...
# df_2 = pd.read_sql(query_2, mini)
# display(df_2)

### Parte 2 (Pregunta 6): Comparación SQL vs Pandas

In [ ]:
# Versión A (Todo SQL)
query_6_sql = """
SELECT ciudad, SUM(total) as ingreso_total
FROM ventas
JOIN clientes ON ventas.cliente_id = clientes.id
GROUP BY ciudad;
"""
df_6_sql = pd.read_sql(query_6_sql, mini)
display(df_6_sql)

In [ ]:
# Versión B (Todo pandas)
ventas_df = pd.read_sql("SELECT * FROM ventas", mini)
clientes_df = pd.read_sql("SELECT * FROM clientes", mini)

merged_df = pd.merge(ventas_df, clientes_df, left_on='cliente_id', right_on='id')
df_6_pandas = merged_df.groupby('ciudad')['total'].sum().reset_index()
display(df_6_pandas)

#### Explicación: ¿Por qué SQL es más eficiente para agregaciones?
En la **Versión A (SQL)**, el motor de la base de datos realiza el `JOIN` y la agregación internamente. A la memoria de Python (pandas) solo se transfiere el resultado final (muy pocas filas, una por ciudad). 

En la **Versión B (pandas)**, se transfieren *todas* las filas de las tablas `ventas` y `clientes` desde la base de datos a la memoria RAM, para luego hacer el merge y groupby en Python. Si hay millones de transacciones, todas viajan por la red o colapsan la memoria innecesariamente. SQL está altamente optimizado para procesar datos donde residen y reducir la transferencia.

### Pregunta 7: Ventas diarias vs Promedio móvil 7 (Matplotlib)

In [ ]:
query_7 = """
SELECT fecha, SUM(total) as ventas_dia
FROM ventas
GROUP BY fecha
ORDER BY fecha;
"""
df_7 = pd.read_sql(query_7, mini)

# Calcular promedio móvil 7 (usando pandas)
df_7['promedio_movil_7'] = df_7['ventas_dia'].rolling(window=7, min_periods=1).mean()

# Graficar
plt.figure(figsize=(10, 6))
plt.plot(pd.to_datetime(df_7['fecha']), df_7['ventas_dia'], label='Ventas Diarias', alpha=0.6)
plt.plot(pd.to_datetime(df_7['fecha']), df_7['promedio_movil_7'], label='Promedio Móvil 7', color='red', linewidth=2)
plt.title('Ventas Diarias vs Promedio Móvil (7 días)')
plt.xlabel('Fecha')
plt.ylabel('Ventas')
plt.legend()
plt.grid(True)
plt.show()

### Parte 4: Tabla de Decisión (SQL vs Pandas)

| Tarea | Preferencia (SQL / pandas) | Justificación |
| :--- | :--- | :--- |
| **Filtrar 50 M de filas** | SQL | SQL filtra los datos en el motor antes de enviarlos, evitando desbordar la memoria RAM. |
| **Unir 4 tablas (JOINs)** | SQL | El motor relacional está optimizado con índices y planes de ejecución para cruzar tablas eficientemente. |
| **Calcular Promedios Móviles** | pandas | Pandas tiene funciones vectorizadas robustas como `.rolling()` que simplifican el análisis de series de tiempo complejas. |
| **Graficar Ventas vs Tiempo** | pandas (+ matplotlib) | Las bases de datos relacionales no tienen herramientas nativas de visualización. |
| **Imputar valores nulos con ML** | pandas / Python | Python cuenta con un amplio ecosistema (ej. scikit-learn) para machine learning y modelado estadístico avanzado. |

## Parte 5: Base Personal y Retos (R1 a R5)

In [ ]:
# Crear conexión a la base personal usando tu código
conn = crear_base(MI_CODIGO)

In [ ]:
# R1: Top 3 clientes por gasto total (nombre, ciudad, total)
query_r1 = """
SELECT c.nombre, c.ciudad, SUM(v.total) as gasto_total
FROM clientes c
JOIN ventas v ON c.id = v.cliente_id
GROUP BY c.id, c.nombre, c.ciudad
ORDER BY gasto_total DESC
LIMIT 3;
"""
display(pd.read_sql(query_r1, conn))

In [ ]:
# R2: Ingreso total por ciudad y categoría
query_r2 = """
SELECT c.ciudad, p.categoria, SUM(v.total) as ingreso_total
FROM ventas v
JOIN clientes c ON v.cliente_id = c.id
JOIN productos p ON v.producto_id = p.id
GROUP BY c.ciudad, p.categoria
ORDER BY c.ciudad, ingreso_total DESC;
"""
display(pd.read_sql(query_r2, conn))

In [ ]:
# R3: Día de la semana con mayor ingreso promedio
# strftime('%w', fecha) retorna 0-6 (0 es Domingo)
query_r3 = """
SELECT strftime('%w', fecha) as dia_semana, AVG(total_dia) as ingreso_promedio
FROM (
    SELECT fecha, SUM(total) as total_dia
    FROM ventas
    GROUP BY fecha
)
GROUP BY dia_semana
ORDER BY ingreso_promedio DESC
LIMIT 1;
"""
display(pd.read_sql(query_r3, conn))

In [ ]:
# R4: Conteo de ventas categoría "Postre" por cliente (incluso los de 0 ventas, usando LEFT JOIN)
query_r4 = """
SELECT c.nombre, COUNT(v.id) as compras_postres
FROM clientes c
LEFT JOIN ventas v ON c.id = v.cliente_id 
                   AND v.producto_id IN (SELECT id FROM productos WHERE categoria = 'Postre')
GROUP BY c.id, c.nombre
ORDER BY compras_postres DESC;
"""
display(pd.read_sql(query_r4, conn))

In [ ]:
# R5: Producto más vendido en unidades por cada categoría (usando ROW_NUMBER() OVER(PARTITION BY...))
query_r5 = """
WITH VentasProducto AS (
    SELECT p.categoria, p.nombre, SUM(v.cantidad) as total_unidades
    FROM ventas v
    JOIN productos p ON v.producto_id = p.id
    GROUP BY p.categoria, p.nombre
),
RankedVentas AS (
    SELECT categoria, nombre, total_unidades,
           ROW_NUMBER() OVER(PARTITION BY categoria ORDER BY total_unidades DESC) as rn
    FROM VentasProducto
)
SELECT categoria, nombre, total_unidades
FROM RankedVentas
WHERE rn = 1;
"""
display(pd.read_sql(query_r5, conn))

In [ ]:
# Ejecución del experimento a escala
experimento_escala(MI_CODIGO)

## Parte 6: El Agente Don Mario (IA)

### Análisis del Agente Reflejo
*¿Por qué pierde dinero el agente reflejo?*
El agente reflejo lanza promociones reaccionando únicamente al ingreso del día anterior comparado con el promedio general. Debido a la estacionalidad (por ejemplo, los lunes siempre venden menos que los viernes), el agente observa un bajón normal (típico de un lunes) y entra en pánico, lanzando una promoción al día siguiente (martes) cuando la demanda suele ser baja de todas formas. Otorgar un 25% de descuento en días donde no habrá gran volumen de ventas no logra compensar el costo fijo de la promoción (ej. 100 soles), generando así pérdidas netas. El agente falla porque carece de contexto sobre qué día de la semana está operando.

In [ ]:
# Reto 1 (Agente Modelo): Aislar el efecto de la estacionalidad
def decidir_modelo(p):
    """
    Agente basado en modelos que aprovecha la memoria del estado.
    Compara el ingreso de ayer con el promedio histórico del *mismo* día de la semana.
    Si está un 15% por debajo, lanza promoción.
    """
    ingreso_ayer = p.get('ingreso_ayer')
    dia_semana_ayer = p.get('dia_semana_ayer')
    df_historico = p.get('historico') # Asumimos que percibir() nos pasa un DataFrame con el historial
    
    if df_historico is None or df_historico.empty:
        return "NADA" # Sin historial no podemos modelar correctamente
        
    # Filtrar el historial solo para el mismo día de la semana
    df_mismo_dia = df_historico[df_historico['dia_semana'] == dia_semana_ayer]
    
    if len(df_mismo_dia) == 0:
        return "NADA"
        
    promedio_dia_semana = df_mismo_dia['ingreso'].mean()
    
    # Si el ingreso de ayer estuvo un 15% por debajo del promedio para ese día específico
    if ingreso_ayer < (promedio_dia_semana * 0.85):
        return "PROMO"
    
    return "NADA"

In [ ]:
# Reto 2 (Agente Objetivo): Mantener ingresos altos a fin de mes/semana

# Modificación teórica a la función percibir()
# Para que este agente funcione, la función percibir() debe nutrir el estado con 
# variables de progreso hacia una meta. Por ejemplo:
def percibir_idea(bd_conn):
    # ... (código que lee la base de datos) ...
    # estado['ingreso_acumulado_mes'] = SELECT SUM(total) FROM ventas WHERE strftime('%Y-%m', fecha) = mes_actual
    # estado['dias_para_cierre'] = (ultimo_dia_mes - dia_actual)
    # estado['meta_mes'] = 20000
    pass

def decidir_objetivo(p):
    """
    Agente basado en objetivos: Dispara promoción solo si estamos rezagados de la meta.
    """
    dias_faltantes = p.get('dias_para_cierre', 30)
    acumulado = p.get('ingreso_acumulado_mes', 0)
    meta = p.get('meta_mes', 10000)
    
    # Si ya superamos o igualamos la meta, ahorramos el costo de la promoción
    if acumulado >= meta:
        return "NADA"
        
    if dias_faltantes > 0:
        ritmo_actual = acumulado / (30 - dias_faltantes) if (30 - dias_faltantes) > 0 else acumulado
        ritmo_necesario = (meta - acumulado) / dias_faltantes
        
        # Si necesitamos ir más rápido de lo que vamos actualmente (estamos rezagados)
        if ritmo_necesario > ritmo_actual:
            return "PROMO"
            
    return "NADA"

In [ ]:
# Ejecución del torneo
agentes_personalizados = {
    'Agente Modelo': decidir_modelo,
    'Agente Objetivo': decidir_objetivo
}

# torneo() medirá el performance de tus agentes contra los predeterminados
torneo(MI_CODIGO, agentes_personalizados)

### La Base de Datos como Entorno del Agente

En el diseño clásico de Inteligencia Artificial (Russell & Norvig), un agente interactúa con su entorno a través de sensores y actuadores. En este laboratorio:

- **El Entorno:** Es la **base de datos SQLite** (que almacena el estado de las ventas, el clima, clientes y productos del día a día).
- **Los Sensores:** Son las **consultas `SELECT`** (ejecutadas dentro de la función `percibir()`). A través de ellas, el agente "ve" u "observa" cómo le fue ayer, qué día de la semana es, etc.
- **Los Actuadores:** Son las sentencias **`INSERT` / `UPDATE`** (o llamadas a funciones que las ejecutan). Al decidir lanzar una promoción, el agente inyecta una acción en el sistema, lo cual afecta el volumen de ventas y los ingresos que se registrarán en la base de datos al día siguiente.